# Pandas Practice (Titanic data set)

Hands-on exercises to go **with** the crash-review notebook (`Pandas_quick_recup.ipynb`).

Read the task, write your own solution in the *your code* cell, then expand/run the **Solution** cell to check yourself. Don't peek too early :)

Dataset: `data/train_and_test2.csv` (same one used in the recap).

| # | Topic | Skill |
|---|-------|-------|
| 1 | Load & inspect | read_csv, dtypes, isnull, value_counts |
| 2 | Filtering | boolean masks, median, survival share |
| 3 | Cleaning & features | fillna, groupby-transform, pd.cut, assign |
| 4 | groupby & pivot | pivot_table, groupby + sort |
| 5 | Check yourself | full mini-pipeline |

---
## Setup
Run this once. Tasks 1 - 4 build on `df`; Task 5 is self-contained (loads from scratch).

In [1]:
import pandas as pd
import numpy as np

DATA = r"data\train_and_test2.csv"

cols = ['Passengerid', 'Age', 'Fare', 'Sex', 'sibsp', 'Parch', 'Pclass', 'Embarked', '2urvived']
df = (
    pd.read_csv(DATA)[cols]
    .rename(columns={'sibsp': 'SibSp', '2urvived': 'Survived'})
)
df['Sex_label'] = df['Sex'].map({0: 'male', 1: 'female'})
df.head()

,Passengerid,Age,Fare,Sex,SibSp,Parch,Pclass,Embarked,Survived,Sex_label
0,1,22.0,7.2500,0,1,0,3,2.0,0,male
1,2,38.0,71.2833,1,1,0,1,0.0,1,female
2,3,26.0,7.9250,1,0,0,3,2.0,1,female
3,4,35.0,53.1000,1,1,0,1,2.0,1,female
4,5,35.0,8.0500,0,0,0,3,2.0,0,male


---
## Task 1: Load & inspect

Using `df` from the setup cell, print:
1. the shape (rows, columns),
2. the dtype of each column,
3. how many missing values each column has (only show columns that actually have any),
4. the overall survival rate as a percentage rounded to 1 decimal.

In [ ]:
# Your code:


In [ ]:
# Solution
print('Shape:', df.shape)
print()
print(df.dtypes)
print()

missing = df.isnull().sum()
print('Missing values (only columns with NaN):')
print(missing[missing > 0])
print()

rate = df['Survived'].mean() * 100
print(f'Overall survival rate: {rate:.1f}%')

---
## Task 2: Filtering with boolean masks

1. How many **female** passengers in **Pclass 1** paid a `Fare` **above the overall median fare**?
2. What share of *those* passengers survived (as a %)?

Hint: build separate masks and combine them with `&` (remember the parentheses).

In [ ]:
# Your code:


In [ ]:
# Solution
median_fare = df['Fare'].median()

mask = (df['Sex_label'] == 'female') & (df['Pclass'] == 1) & (df['Fare'] > median_fare)
subset = df[mask]

print(f'Median fare: {median_fare:.2f}')
print(f'Rich 1st-class women: {len(subset)}')
print(f'Their survival rate: {subset["Survived"].mean() * 100:.1f}%')

# Same thing with query():
# df.query('Sex_label == "female" and Pclass == 1 and Fare > @median_fare')

---
## Task 3 : Cleaning & feature engineering

Work on a **copy** `df3 = df.copy()` and:
1. fill missing `Embarked` with the column's **mode**,
2. fill missing `Age` with the **median within each `Pclass`** group,
3. add `FamilySize = SibSp + Parch + 1`,
4. add `IsAlone` (1 if the passenger travels alone, else 0),
5. add `AgeGroup` with bins: `child` (<18), `adult` (<60), `senior` (60+).

Finally, confirm there are **no missing values** left in `Age` and `Embarked`.

In [ ]:
# Your code:


In [ ]:
# Solution
df3 = df.copy()

# 1. Embarked -> mode
df3['Embarked'] = df3['Embarked'].fillna(df3['Embarked'].mode()[0])

# 2. Age -> median within Pclass
df3['Age'] = df3.groupby('Pclass')['Age'].transform(lambda x: x.fillna(x.median()))

# 3-5. features
df3['FamilySize'] = df3['SibSp'] + df3['Parch'] + 1
df3['IsAlone'] = np.where(df3['FamilySize'] == 1, 1, 0)
df3['AgeGroup'] = pd.cut(df3['Age'], bins=[0, 18, 60, 200], labels=['child', 'adult', 'senior'])

print('Missing in Age:', df3['Age'].isnull().sum())
print('Missing in Embarked:', df3['Embarked'].isnull().sum())
print()
display(df3[['Age', 'AgeGroup', 'FamilySize', 'IsAlone']].head())

---
## Task 4 : groupby & pivot_table

1. Build a `pivot_table` of **survival rate** with `Pclass` as rows, `Sex_label` as columns, and a `margins=True` totals row/column. Round to 3 decimals.
2. Which `Embarked` port has the **highest average `Fare`**? (use `groupby` + `sort_values`)

Embarked codes: `0 = S`, `1 = C`, `2 = Q`.

In [ ]:
# Your code:


In [ ]:
# Solution
pt = pd.pivot_table(
    df,
    values='Survived',
    index='Pclass',
    columns='Sex_label',
    aggfunc='mean',
    margins=True,
).round(3)
print('Survival rate by Pclass x Sex:')
display(pt)

port_map = {0: 'S', 1: 'C', 2: 'Q'}
fare_by_port = (
    df.assign(Port=df['Embarked'].map(port_map))
    .groupby('Port')['Fare'].mean()
    .sort_values(ascending=False)
    .round(2)
)
print('\nAverage fare by embark port:')
print(fare_by_port)
print('\nHighest:', fare_by_port.idxmax())

---
## Task 5 : Check yourself (full mini-pipeline)

Do this one **from scratch** (reload the CSV yourself):

1. Load the dataset (`data/train_and_test2.csv`).
2. Keep columns: `Passengerid, Age, Fare, Sex, sibsp, Parch, Pclass, 2urvived`.
3. Rename `sibsp -> SibSp`, `2urvived -> Survived`.
4. Fill missing `Age` with the median within each `Pclass`.
5. Add `FamilySize = SibSp + Parch + 1` and `IsAlone` (1 if alone).
6. Show the survival rate (%) per `Pclass`, sorted high to low.
7. Find the top-3 `Pclass` + `Sex` combinations by survival rate.

In [ ]:
# Your code:
df_task = pd.read_csv(r"data\train_and_test2.csv")


In [ ]:
# Solution
df_sol = (
    pd.read_csv(r"data\train_and_test2.csv")
    [['Passengerid', 'Age', 'Fare', 'Sex', 'sibsp', 'Parch', 'Pclass', '2urvived']]
    .rename(columns={'2urvived': 'Survived', 'sibsp': 'SibSp'})
    .assign(
        Age        = lambda x: x.groupby('Pclass')['Age'].transform(
                                  lambda g: g.fillna(g.median())),
        FamilySize = lambda x: x['SibSp'] + x['Parch'] + 1,
        IsAlone    = lambda x: (x['SibSp'] + x['Parch'] == 0).astype(int),
        Sex_label  = lambda x: x['Sex'].map({0: 'male', 1: 'female'}),
    )
)

print('=== 6. Survival rate by Pclass ===')
print(
    df_sol.groupby('Pclass')['Survived']
    .mean().mul(100).round(1)
    .sort_values(ascending=False)
    .rename('survival_%')
)

print('\n=== 7. Top-3 Pclass x Sex ===')
print(
    df_sol.groupby(['Pclass', 'Sex_label'])['Survived']
    .mean().mul(100).round(1)
    .sort_values(ascending=False)
    .head(3)
    .rename('survival_%')
)